In [70]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [71]:
import pandas as pd
import numpy as np
import joblib

In [72]:
match_data = pd.read_csv("../data/match_data.csv")
match_info_data = pd.read_csv("../data/match_info_data.csv")

C:\Users\Nithish\AppData\Local\Temp\ipykernel_13732\2981192924.py:1: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  match_data = pd.read_csv("../data/match_data.csv")


In [73]:
print(match_data.shape)
print(match_info_data.shape)

(243817, 23)
(1024, 18)


In [74]:
merged_data = match_data.merge(
    match_info_data,
    left_on="match_id",
    right_on="id"
)

In [75]:
merged_data.shape

(243817, 41)

In [76]:
merged_data["total_runs"] = (
    merged_data["runs_off_bat"] +
    merged_data["extras"]
)

In [77]:
merged_data[
    ["runs_off_bat", "extras", "total_runs"]
].head(10)

,runs_off_bat,extras,total_runs
0,0,0,0
1,0,0,0
2,1,0,1
3,1,0,1
4,1,0,1
5,1,0,1
6,0,0,0
7,1,0,1
8,1,0,1
9,1,0,1


In [78]:
merged_data["current_score"] = (
    merged_data
    .groupby(["match_id", "innings"])["total_runs"]
    .cumsum()
)

In [79]:
merged_data[
    ["match_id", "innings", "ball", "total_runs", "current_score"]
].head(15)

,match_id,innings,ball,total_runs,current_score
0,1370353,1,0.1,0,0
1,1370353,1,0.2,0,0
2,1370353,1,0.3,1,1
3,1370353,1,0.4,1,2
4,1370353,1,0.5,1,3
5,1370353,1,0.6,1,4
6,1370353,1,1.1,0,4
7,1370353,1,1.2,1,5
8,1370353,1,1.3,1,6
9,1370353,1,1.4,1,7


In [81]:
merged_data["final_score"] = (
    merged_data
    .groupby(["match_id", "innings"])["current_score"]
    .transform("max")
)

In [82]:
(merged_data["current_score"] > merged_data["final_score"]).sum()

0

In [83]:
merged_data[
    [
        "match_id",
        "innings",
        "ball",
        "current_score",
        "final_score"
    ]
].head(20)

,match_id,innings,ball,current_score,final_score
0,1370353,1,0.1,0,214
1,1370353,1,0.2,0,214
2,1370353,1,0.3,1,214
3,1370353,1,0.4,2,214
4,1370353,1,0.5,3,214
5,1370353,1,0.6,4,214
6,1370353,1,1.1,4,214
7,1370353,1,1.2,5,214
8,1370353,1,1.3,6,214
9,1370353,1,1.4,7,214


In [84]:
score_data = merged_data[merged_data["innings"] == 1].copy()

In [85]:
score_data["innings"].value_counts()

innings
1    126125
Name: count, dtype: int64

In [86]:
score_data["ball"].head(30)

0     0.1
1     0.2
2     0.3
3     0.4
4     0.5
5     0.6
6     1.1
7     1.2
8     1.3
9     1.4
10    1.5
11    1.6
12    2.1
13    2.2
14    2.3
15    2.4
16    2.5
17    2.6
18    3.1
19    3.2
20    3.3
21    3.4
22    3.5
23    3.6
24    4.1
25    4.2
26    4.3
27    4.4
28    4.5
29    4.6
Name: ball, dtype: float64

In [87]:
score_data["ball"].tail(30)

243686    15.1
243687    15.2
243688    15.3
243689    15.4
243690    15.5
243691    15.6
243692    16.1
243693    16.2
243694    16.3
243695    16.4
243696    16.5
243697    16.6
243698    17.1
243699    17.2
243700    17.3
243701    17.4
243702    17.5
243703    17.6
243704    18.1
243705    18.2
243706    18.3
243707    18.4
243708    18.5
243709    18.6
243710    19.1
243711    19.2
243712    19.3
243713    19.4
243714    19.5
243715    19.6
Name: ball, dtype: float64

In [88]:
score_data = score_data[
    score_data["ball"].astype(str).str.endswith(".6")
].copy()

In [89]:
score_data["ball"].head(20)

5       0.6
11      1.6
17      2.6
23      3.6
29      4.6
35      5.6
41      6.6
47      7.6
53      8.6
59      9.6
65     10.6
71     11.6
79     12.6
85     13.6
91     14.6
97     15.6
103    16.6
109    17.6
115    18.6
121    19.6
Name: ball, dtype: float64

In [90]:
score_data.shape

(20252, 44)

In [91]:
score_data["wickets_lost"] = (
    score_data.groupby("match_id")["player_dismissed"]
    .transform(lambda x: x.notna().cumsum())
)

In [92]:
score_data[
    ["ball", "player_dismissed", "wickets_lost"]
].head(30)

,ball,player_dismissed,wickets_lost
5,0.6,NaN,0
11,1.6,NaN,0
17,2.6,NaN,0
23,3.6,NaN,0
29,4.6,NaN,0
35,5.6,NaN,0
41,6.6,Shubman Gill,1
47,7.6,NaN,1
53,8.6,NaN,1
59,9.6,NaN,1


In [93]:
score_data["overs_completed"] = score_data["ball"].astype(int) + 1

In [94]:
score_data[
    ["ball", "overs_completed"]
].head(20)

,ball,overs_completed
5,0.6,1
11,1.6,2
17,2.6,3
23,3.6,4
29,4.6,5
35,5.6,6
41,6.6,7
47,7.6,8
53,8.6,9
59,9.6,10


In [95]:
score_data["current_run_rate"] = (
    score_data["current_score"] /
    score_data["overs_completed"]
)

In [96]:
score_data[
    [
        "current_score",
        "overs_completed",
        "current_run_rate"
    ]
].head(20)

,current_score,overs_completed,current_run_rate
5,4,1,4.000000
11,8,2,4.000000
17,24,3,8.000000
23,38,4,9.500000
29,49,5,9.800000
35,62,6,10.333333
41,67,7,9.571429
47,72,8,9.000000
53,80,9,8.888889
59,86,10,8.600000


In [97]:
projected_score_data = score_data[
    [
        "batting_team",
        "bowling_team",
        "venue_x",
        "current_score",
        "wickets_lost",
        "overs_completed",
        "current_run_rate",
        "final_score"
    ]
].copy()

In [98]:
projected_score_data.head()

,batting_team,bowling_team,venue_x,current_score,wickets_lost,overs_completed,current_run_rate,final_score
5,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",4,0,1,4.0,214
11,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",8,0,2,4.0,214
17,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",24,0,3,8.0,214
23,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",38,0,4,9.5,214
29,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",49,0,5,9.8,214


In [99]:
projected_score_data.shape

(20252, 8)

In [100]:
X = projected_score_data.drop("final_score", axis=1)
y = projected_score_data["final_score"]

In [101]:
X.head()

,batting_team,bowling_team,venue_x,current_score,wickets_lost,overs_completed,current_run_rate
5,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",4,0,1,4.0
11,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",8,0,2,4.0
17,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",24,0,3,8.0
23,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",38,0,4,9.5
29,Gujarat Titans,Chennai Super Kings,"Narendra Modi Stadium, Ahmedabad",49,0,5,9.8


In [102]:
y.head()

5     214
11    214
17    214
23    214
29    214
Name: final_score, dtype: int64

In [103]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [104]:
print(X_train.shape)
print(X_test.shape)

(16201, 7)
(4051, 7)


In [105]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ["batting_team", "bowling_team", "venue_x"]
        )
    ],
    remainder="passthrough"
)

In [106]:
preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['batting_team', 'bowling_team', 'venue_x'])])

In [107]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

In [108]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

In [110]:
lr_model.fit(X_train, y_train)

c:\Users\Nithish\OneDrive\Desktop\IPL-Analytics-Pro\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['batting_team',
                                                   'bowling_team',
                                                   'venue_x'])])),
                ('model', LinearRegression())])

In [111]:
y_pred_lr = lr_model.predict(X_test)

In [112]:
print("Linear Regression")
print("MAE :", mean_absolute_error(y_test, y_pred_lr))
print("R2  :", r2_score(y_test, y_pred_lr))

Linear Regression
MAE : 15.208949459651397
R2  : 0.5199714601380836


In [113]:
from sklearn.ensemble import RandomForestRegressor

In [114]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

In [115]:
rf_model.fit(X_train, y_train)

c:\Users\Nithish\OneDrive\Desktop\IPL-Analytics-Pro\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['batting_team',
                                                   'bowling_team',
                                                   'venue_x'])])),
                ('model',
                 RandomForestRegressor(max_depth=15, min_samples_leaf=2,
                                       n_jobs=-1, random_state=42))])

In [116]:
y_pred_rf = rf_model.predict(X_test)

In [117]:
print("Random Forest")
print("MAE :", mean_absolute_error(y_test, y_pred_rf))
print("R2  :", r2_score(y_test, y_pred_rf))

Random Forest
MAE : 13.620874852471191
R2  : 0.5966072664248079


In [118]:
input_df = pd.DataFrame({
    "batting_team": ["Chennai Super Kings"],
    "bowling_team": ["Deccan Chargers"],
    "venue_x": ["Arun Jaitley Stadium"],
    "current_score": [234],
    "wickets_lost": [4],
    "overs_completed": [15],
    "current_run_rate": [15.6]
})

In [119]:
rf_model.predict(input_df)

array([214.95865731])

In [120]:
rf_model.predict(input_df)

array([214.95865731])

In [121]:
score_data[
    (score_data["overs_completed"] == 15) &
    (score_data["current_score"] >= 220)
][["current_score", "final_score"]]

,current_score,final_score


In [122]:
len(
    score_data[
        (score_data["overs_completed"] == 15) &
        (score_data["current_score"] >= 220)
    ]
)

0

In [123]:
import joblib

joblib.dump(rf_model, "../model/projected_score_model.pkl")

['../model/projected_score_model.pkl']

In [124]:
import os

print(os.path.abspath("../model/projected_score_model.pkl"))

c:\Users\Nithish\OneDrive\Desktop\IPL-Analytics-Pro\model\projected_score_model.pkl


In [125]:
import os

os.path.exists("../model/projected_score_model.pkl")

True